In [13]:
import csv, json, os, urllib.parse, urllib.request

BASE = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
params = {
    "format": "json",
    "no_aggs": "true",
    "no_highlight": "true",
    "product": "Mortgage",
    "has_narrative": "true",
    "date_received_min": "2025-01-01",
    "date_received_max": "2026-04-15",
    "consumer_consent_provided": "Consent provided",
}
q = urllib.parse.urlencode(params)
with urllib.request.urlopen(f"{BASE}?{q}", timeout=300) as r:
    data = json.load(r)
if isinstance(data, dict) and data.get("error"):
    raise RuntimeError(data["error"])
rows = [h["_source"] for h in data]
if not rows:
    raise RuntimeError("No rows returned")

cols = ["complaint_id", "date_received", "issue", "sub_issue", "complaint_what_happened"]
rows_out = [{c: (r.get(c) or "") for c in cols} for r in rows]

os.makedirs("data", exist_ok=True)
out = "data/mortgage_2025_narrative_consent.csv"
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(rows_out)
len(rows_out), out

(16209, 'data/mortgage_2025_narrative_consent.csv')

In [14]:
import re

import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TARGET_COL = "issue"
TEXT_COL = "complaint_what_happened"

mortgage_df = pd.read_csv("data/mortgage_2025_narrative_consent.csv")

In [15]:
mortgage_df.head()

,complaint_id,date_received,issue,sub_issue,complaint_what_happened
0,15844200,2025-09-10T12:00:00-05:00,Struggling to pay mortgage,Trying to communicate with the company to fix ...,"Dear CFPB, We are filing a formal complaint ag..."
1,16329523,2025-10-02T12:00:00-05:00,Trouble during payment process,Payment process,This company continues to keep reporting and m...
2,13427712,2025-05-09T12:00:00-05:00,Trouble during payment process,"Escrow, taxes, or insurance",I had XXXX mortgages with CMG financial. The f...
3,11466691,2025-01-10T12:00:00-05:00,Struggling to pay mortgage,Trying to communicate with the company to fix ...,I currently have an ongoing mortgage account w...
4,17365063,2025-11-19T12:00:00-05:00,Trouble during payment process,Payment process,I transferred {$300000.00} from my checking ac...


In [16]:
mortgage_df.count()

complaint_id               16209
date_received              16209
issue                      16209
sub_issue                  16198
complaint_what_happened    16209
dtype: int64

In [17]:
# Step 5: preprocess text and create locked train/holdout datasets

def normalize_text(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

work_df = mortgage_df.copy()
work_df[TEXT_COL] = work_df[TEXT_COL].fillna("").astype(str)
work_df["text_norm"] = work_df[TEXT_COL].map(normalize_text)

initial_n = len(work_df)

# Remove obvious unusable rows: empty text/label
work_df = work_df[(work_df["text_norm"] != "") & (work_df[TARGET_COL].notna())]
non_empty_n = len(work_df)

# Obvious spam heuristics (kept conservative)
spam_pattern = r"^(test|testing|asdf|lorem ipsum|n/?a|none|spam)+$"
work_df["is_obvious_spam"] = (
    work_df["text_norm"].str.fullmatch(spam_pattern, na=False)
    | (~work_df["text_norm"].str.contains(r"[a-z]", regex=True))
)
work_df = work_df[~work_df["is_obvious_spam"]].copy()
no_spam_n = len(work_df)

# Remove duplicates by id and then by normalized text
work_df = work_df.drop_duplicates(subset=["complaint_id"], keep="first")
work_df = work_df.drop_duplicates(subset=["text_norm"], keep="first").reset_index(drop=True)
clean_n = len(work_df)

# Requested fixed dataset sizes
N_TRAIN = 15000
N_HOLDOUT = 1000
N_REQUIRED = N_TRAIN + N_HOLDOUT

if clean_n < N_REQUIRED:
    raise ValueError(f"Need at least {N_REQUIRED} clean rows, found {clean_n}")

# Lock holdout early and keep exactly 15,000 training rows
remaining_df, holdout_df = train_test_split(
    work_df,
    test_size=N_HOLDOUT,
    stratify=work_df[TARGET_COL],
    random_state=RANDOM_STATE,
)

train_df, _discard_df = train_test_split(
    remaining_df,
    train_size=N_TRAIN,
    stratify=remaining_df[TARGET_COL],
    random_state=RANDOM_STATE,
)

columns_to_save = [
    "complaint_id",
    "date_received",
    "issue",
    "sub_issue",
    "complaint_what_happened",
]

train_df = train_df[columns_to_save].reset_index(drop=True)
holdout_df = holdout_df[columns_to_save].reset_index(drop=True)

train_out = "data/mortgage_train_15000.csv"
holdout_out = "data/mortgage_holdout_1000.csv"
train_df.to_csv(train_out, index=False, encoding="utf-8")
holdout_df.to_csv(holdout_out, index=False, encoding="utf-8")


def split_summary(df: pd.DataFrame, name: str) -> pd.Series:
    text_lens = df[TEXT_COL].astype(str).str.len()
    return pd.Series(
        {
            "split": name,
            "rows": len(df),
            "classes": df[TARGET_COL].nunique(),
            "avg_chars": round(text_lens.mean(), 1),
            "median_chars": int(text_lens.median()),
            "p95_chars": int(text_lens.quantile(0.95)),
        }
    )

summary_df = pd.DataFrame(
    [
        split_summary(work_df, "clean_total"),
        split_summary(train_df, "train_15000"),
        split_summary(holdout_df, "holdout_1000"),
    ]
)

print("Cleaning summary")
print(
    {
        "initial_rows": initial_n,
        "after_non_empty": non_empty_n,
        "after_spam_filter": no_spam_n,
        "after_dedup": clean_n,
    }
)

print("\nSaved files")
print({"train_csv": train_out, "holdout_csv": holdout_out})

print("\nSplit sizes")
print({"train": len(train_df), "holdout": len(holdout_df)})

print("\nLength/basic summary")
display(summary_df)

print("\nLabel distributions (proportion)")
label_dist = pd.concat(
    {
        "train": train_df[TARGET_COL].value_counts(normalize=True),
        "holdout": holdout_df[TARGET_COL].value_counts(normalize=True),
    },
    axis=1,
).fillna(0)
display(label_dist.sort_values(by="train", ascending=False).head(15))

Cleaning summary
{'initial_rows': 16209, 'after_non_empty': 16209, 'after_spam_filter': 16209, 'after_dedup': 16200}

Saved files
{'train_csv': 'data/mortgage_train_15000.csv', 'holdout_csv': 'data/mortgage_holdout_1000.csv'}

Split sizes
{'train': 15000, 'holdout': 1000}

Length/basic summary


,split,rows,classes,avg_chars,median_chars,p95_chars
0,clean_total,16200,10,1757.4,1277,4747
1,train_15000,15000,10,1754.8,1276,4743
2,holdout_1000,1000,10,1789.3,1290,4710



Label distributions (proportion)


,train,holdout
Trouble during payment process,0.520933,0.521
Struggling to pay mortgage,0.228067,0.228
Applying for a mortgage or refinancing an existing mortgage,0.108800,0.109
Closing on a mortgage,0.090400,0.090
Incorrect information on your report,0.028133,0.028
Problem with a company's investigation into an existing problem,0.017733,0.018
Improper use of your report,0.003133,0.003
Credit monitoring or identity theft protection services,0.001333,0.001
Unable to get your credit report or credit score,0.000800,0.001
Problem with fraud alerts or security freezes,0.000667,0.001
